In [0]:
# configure spark and import required libraries
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

from pyspark.sql.functions import col, trim, upper, sum as spark_sum, regexp_replace, to_date
from pyspark.sql.types import DecimalType, IntegerType

In [0]:
# ingest raw sales data
df_sales_cleaned = spark.read.format("csv").option("header", "true").option("inferSchema", True).option("sep", "\t").load("/Volumes/final_project/bronze/data/Sales_*.csv")
display(df_sales_cleaned)

In [0]:
df_sales_cleaned.printSchema()

In [0]:
# text standardization: trim whitespace
df_sales_cleaned = df_sales_cleaned.select([trim(col(c)).alias(c) for c in df_sales_cleaned.columns])

In [0]:
# standardize column names
df_sales_cleaned = df_sales_cleaned \
    .withColumnRenamed("SalesOrderNumber", "sales_order_number") \
    .withColumnRenamed("OrderDate", "order_date") \
    .withColumnRenamed("ProductKey", "product_key") \
    .withColumnRenamed("ResellerKey", "reseller_key") \
    .withColumnRenamed("EmployeeKey", "employee_key") \
    .withColumnRenamed("SalesTerritoryKey", "sales_territory_key") \
    .withColumnRenamed("Quantity", "quantity") \
    .withColumnRenamed("Unit Price", "unit_price") \
    .withColumnRenamed("Sales", "sales") \
    .withColumnRenamed("Cost", "cost")

In [0]:
# text standardization: uppercase identifiers
# ensure all sales order numbers follow a uniform uppercase format.
df_sales_cleaned = df_sales_cleaned.withColumn("sales_order_number", upper(col("sales_order_number")))

In [0]:
# clean currency strings
df_sales_cleaned = df_sales_cleaned \
    .withColumn("unit_price", regexp_replace(col("unit_price"), "[$,]", "")) \
    .withColumn("sales",      regexp_replace(col("sales"),      "[$,]", "")) \
    .withColumn("cost",       regexp_replace(col("cost"),       "[$,]", ""))

display(df_sales_cleaned)

In [0]:
# data profiling
total_rows = df_sales_cleaned.count()
distinct_rows = df_sales_cleaned.distinct().count()

print(f"Total rows:    {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicates:    {total_rows - distinct_rows}")

# handle duplicates if exist
df_sales_cleaned = df_sales_cleaned.dropDuplicates()

In [0]:
# enforce strict data types
df_sales_cleaned = df_sales_cleaned \
    .withColumn("order_date",         to_date(col("order_date"), "EEEE, MMMM d, yyyy")) \
    .withColumn("product_key",        col("product_key").cast(IntegerType())) \
    .withColumn("reseller_key",       col("reseller_key").cast(IntegerType())) \
    .withColumn("employee_key",       col("employee_key").cast(IntegerType())) \
    .withColumn("sales_territory_key",col("sales_territory_key").cast(IntegerType())) \
    .withColumn("quantity",           col("quantity").cast(IntegerType())) \
    .withColumn("unit_price",         col("unit_price").cast(DecimalType(10, 2))) \
    .withColumn("sales",              col("sales").cast(DecimalType(10, 2))) \
    .withColumn("cost",               col("cost").cast(DecimalType(10, 2)))

# display schema
df_sales_cleaned.printSchema()
display(df_sales_cleaned)

In [0]:
# Check for null values
null_counts = df_sales_cleaned.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_sales_cleaned.columns
])

display(null_counts)

# Handle missing or invalid values if exist 
df_sales_cleaned = df_sales_cleaned.dropna()

In [0]:
# Validate schema
df_sales_cleaned.printSchema()

- **Complete the Cleaning State**
- **Save as Delta Table**

In [0]:
# Write schema silver layer
df_sales_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.silver.sales")